# Benchmarking the Rust Numerical Computing Examples Against NumPy and scikit-learn

This notebook compares the Rust implementations in this repository with widely used Python scientific-computing libraries. The goal is not just to report timings, but to explain what is being benchmarked, how the benchmark works, and what the results mean for an external reader.

The Rust code is exposed to Python through `pyo3` bindings and installed with `maturin`, which makes it possible to import the crate directly inside Jupyter as `rust_numeric`.

## Functions under test

The notebook focuses on the public Rust implementations that map cleanly to established NumPy or scikit-learn APIs:

- `stats::mean` compared with `numpy.mean`
- `stats::percentile` compared with `numpy.percentile`
- `linalg::matmul` compared with `numpy.matmul`
- `linalg::inv` compared with `numpy.linalg.inv`
- `LinearRegression::fit`, `predict`, and `score` compared with `sklearn.linear_model.LinearRegression`

These functions are implemented in:

- `src/stats.rs`
- `src/linalg.rs`
- `src/linear_regression.rs`

The comparison stays close to the Rust code that already exists in the repository instead of introducing a new optimized Rust implementation just for benchmarking.

In [ ]:
from __future__ import annotations

import gc
import os
import statistics
import sys
from pathlib import Path
from time import perf_counter_ns

NOTEBOOK_DIR = Path.cwd()
MPLCONFIGDIR = NOTEBOOK_DIR / '.mplconfig'
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression as SklearnLinearRegression

import rust_numeric

print(f'Python executable: {sys.executable}')
print(f'NumPy version: {np.__version__}')
print(f'rust_numeric import: {rust_numeric.__name__}')

: 

## Benchmark methodology

The benchmark is designed to be simple, transparent, and repeatable:

1. Inputs are generated once with a fixed random seed.
2. Each function is warmed up before timing begins.
3. Each benchmark is run multiple times and summarized with the median runtime.
4. For the Rust bindings, Python lists are prepared once ahead of time so the timed section focuses on the Rust call itself rather than repeated list conversion.

That last point matters. NumPy and scikit-learn are array-native, while these Rust bindings currently accept Python sequences. The results therefore represent practical notebook usage through the Python binding layer, not a lower-level FFI microbenchmark.

In [ ]:
def benchmark(label, func, *, repeats=9, warmup=2):
    for _ in range(warmup):
        func()

    timings_ms = []
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        for _ in range(repeats):
            start = perf_counter_ns()
            func()
            end = perf_counter_ns()
            timings_ms.append((end - start) / 1_000_000)
    finally:
        if gc_was_enabled:
            gc.enable()

    return {
        'label': label,
        'median_ms': statistics.median(timings_ms),
        'mean_ms': statistics.mean(timings_ms),
        'min_ms': min(timings_ms),
        'max_ms': max(timings_ms),
        'all_ms': timings_ms,
    }


def summarize_pair(operation, rust_result, python_result):
    speed_ratio = python_result['median_ms'] / rust_result['median_ms']
    faster = 'Rust' if speed_ratio > 1 else 'Python'
    factor = speed_ratio if speed_ratio > 1 else 1 / speed_ratio
    return {
        'operation': operation,
        'rust_median_ms': rust_result['median_ms'],
        'python_median_ms': python_result['median_ms'],
        'faster': faster,
        'factor': factor,
    }


def print_summary(rows):
    header = f"{'Operation':<24} {'Rust median (ms)':>16} {'Python median (ms)':>18} {'Faster':>10} {'Factor':>10}"
    print(header)
    print('-' * len(header))
    for row in rows:
        print(
            f"{row['operation']:<24} {row['rust_median_ms']:>16.3f} {row['python_median_ms']:>18.3f} {row['faster']:>10} {row['factor']:>10.2f}"
        )

## Generate benchmark inputs

The data sizes below are large enough to reveal meaningful differences without making the notebook slow to execute in an interactive session.

In [ ]:
rng = np.random.default_rng(42)

vector = rng.normal(size=250_000)
vector_list = vector.tolist()

percentile_q = 75.0

matmul_a = rng.normal(size=(200, 200))
matmul_b = rng.normal(size=(200, 200))
matmul_a_list = matmul_a.tolist()
matmul_b_list = matmul_b.tolist()

inv_matrix = rng.normal(size=(120, 120))
inv_matrix = inv_matrix + np.eye(inv_matrix.shape[0]) * 0.5
inv_matrix_list = inv_matrix.tolist()

x = rng.normal(size=(20_000, 8))
beta = np.array([1.5, -2.0, 0.75, 0.0, 3.0, -1.2, 0.4, 2.2])
noise = rng.normal(scale=0.1, size=x.shape[0])
y = x @ beta + 0.8 + noise

x_list = x.tolist()
y_list = y.tolist()

print('Vector length:', len(vector_list))
print('Matrix multiply shape:', matmul_a.shape, 'x', matmul_b.shape)
print('Inverse matrix shape:', inv_matrix.shape)
print('Regression design matrix shape:', x.shape)

## Sanity check: numerical agreement

Before measuring performance, it is worth confirming that the Rust and Python implementations produce comparable answers.

In [ ]:
rust_lr = rust_numeric.LinearRegression().fit(x_list, y_list)
sk_lr = SklearnLinearRegression().fit(x, y)

agreement = {
    'mean': abs(rust_numeric.mean(vector_list) - float(np.mean(vector))),
    'percentile_75': abs(rust_numeric.percentile(vector_list, percentile_q) - float(np.percentile(vector, percentile_q))),
    'matmul_max_abs_diff': float(np.max(np.abs(np.array(rust_numeric.matmul(matmul_a_list, matmul_b_list)) - (matmul_a @ matmul_b)))),
    'inverse_max_abs_diff': float(np.max(np.abs(np.array(rust_numeric.inverse(inv_matrix_list)) - np.linalg.inv(inv_matrix)))),
    'linear_regression_coef_max_abs_diff': float(np.max(np.abs(np.array(rust_lr.coef_) - sk_lr.coef_))),
    'linear_regression_intercept_abs_diff': abs(rust_lr.intercept_ - float(sk_lr.intercept_)),
}

agreement

## Run the benchmarks

Each comparison pairs a Rust function with the closest mainstream Python equivalent. For the regression model, `fit`, `predict`, and `score` are benchmarked separately because those operations have different cost profiles.

In [ ]:
results = []

mean_rust = benchmark('rust mean', lambda: rust_numeric.mean(vector_list))
mean_numpy = benchmark('numpy mean', lambda: float(np.mean(vector)))
results.append(summarize_pair('mean', mean_rust, mean_numpy))

percentile_rust = benchmark('rust percentile', lambda: rust_numeric.percentile(vector_list, percentile_q), repeats=7)
percentile_numpy = benchmark('numpy percentile', lambda: float(np.percentile(vector, percentile_q)), repeats=7)
results.append(summarize_pair('percentile(75)', percentile_rust, percentile_numpy))

matmul_rust = benchmark('rust matmul', lambda: rust_numeric.matmul(matmul_a_list, matmul_b_list), repeats=5)
matmul_numpy = benchmark('numpy matmul', lambda: matmul_a @ matmul_b, repeats=5)
results.append(summarize_pair('matrix multiply', matmul_rust, matmul_numpy))

inverse_rust = benchmark('rust inverse', lambda: rust_numeric.inverse(inv_matrix_list), repeats=5)
inverse_numpy = benchmark('numpy inverse', lambda: np.linalg.inv(inv_matrix), repeats=5)
results.append(summarize_pair('matrix inverse', inverse_rust, inverse_numpy))

fit_rust = benchmark(
    'rust fit',
    lambda: rust_numeric.LinearRegression().fit(x_list, y_list),
    repeats=5,
)
fit_sklearn = benchmark(
    'sklearn fit',
    lambda: SklearnLinearRegression().fit(x, y),
    repeats=5,
)
results.append(summarize_pair('linear regression fit', fit_rust, fit_sklearn))

trained_rust = rust_numeric.LinearRegression().fit(x_list, y_list)
trained_sklearn = SklearnLinearRegression().fit(x, y)

predict_rust = benchmark('rust predict', lambda: trained_rust.predict(x_list), repeats=7)
predict_sklearn = benchmark('sklearn predict', lambda: trained_sklearn.predict(x), repeats=7)
results.append(summarize_pair('linear regression predict', predict_rust, predict_sklearn))

score_rust = benchmark('rust score', lambda: trained_rust.score(x_list, y_list), repeats=7)
score_sklearn = benchmark('sklearn score', lambda: trained_sklearn.score(x, y), repeats=7)
results.append(summarize_pair('linear regression score', score_rust, score_sklearn))

print_summary(results)

In [ ]:
operations = [row['operation'] for row in results]
rust_times = [row['rust_median_ms'] for row in results]
python_times = [row['python_median_ms'] for row in results]

x_pos = np.arange(len(operations))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_pos - width / 2, rust_times, width, label='Rust via pyo3', color='#1f77b4')
ax.bar(x_pos + width / 2, python_times, width, label='NumPy / scikit-learn', color='#ff7f0e')
ax.set_ylabel('Median runtime (ms)')
ax.set_title('Rust vs. NumPy / scikit-learn benchmark results')
ax.set_xticks(x_pos)
ax.set_xticklabels(operations, rotation=25, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Reading the results

A few patterns are usually worth watching for when you run the notebook:

- If NumPy or scikit-learn is faster on dense linear algebra, that is expected. Those libraries sit on top of mature native backends such as BLAS and LAPACK and have had years of optimization work.
- If some scalar or simpler operations are competitive in Rust, that shows the baseline Rust implementations are already efficient even before introducing specialized crates such as `ndarray`, `nalgebra`, or explicit SIMD tuning.
- The gap between the Rust implementation and NumPy is not just about language choice. It is also about algorithm choice, data layout, and whether the implementation delegates to optimized numerical kernels.

In other words, this notebook is best read as a comparison of the current repository implementations against production-grade Python numerical libraries, not as a claim about the theoretical peak performance of Rust.

## Takeaways

This benchmark gives a concrete, reproducible way to explain the current state of the codebase:

- The Rust functions are now importable directly from Python, which makes the crate much easier to demonstrate from notebooks and teaching materials.
- The performance story is nuanced. Native Python numerical libraries are often extremely fast because the heavy lifting already happens in compiled code.
- The Rust code remains valuable as a clear, inspectable implementation of the underlying numerical ideas, and it creates a strong foundation for future optimization work.

If you want to extend this notebook later, natural next steps would be benchmarking more functions from `stats.rs`, switching the Rust linear algebra code to a library-backed implementation, or adding profiling to explain where the remaining time is spent.